# RFT-0008D-r3mix-r2continue-r16-a100 — continued best R2 Pro4-hint r16

R3 training corpus: one native R1 trace per question, prior Pro4-hint
R2 Qwen rewrite traces, and the newly generated R3 Qwen rewrite traces.
Validation IDs/templates are removed before tokenization. This notebook
reads no leaderboard/test/submission data. It saves TensorBoard and W&B
offline logs, checkpoints, final adapter, corpus manifest, and report.

In [ ]:
# Cell 1 — Fixed A100 BF16 LoRA training stack. Restart only if already imported.
%pip install -q --no-cache-dir "transformers==4.52.3" "peft==0.16.0" "accelerate==1.7.0" "datasets==3.6.0" "bitsandbytes==0.46.0" "tensorboard~=2.19.0" "wandb>=0.19,<1" "protobuf<6"
print("[SETUP] complete")

In [ ]:
# Cell 2 — Reproducible config, Drive paths, and safe immutable source discovery.
import gc, hashlib, inspect, json, math, os, platform, re, subprocess, time, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, torch
from google.colab import drive

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_MODE"] = "offline"; os.environ["WANDB_PROJECT"] = "deep-learning-challenge-2026"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"; MODEL_REVISION = "main"
RUN_ID = "RFT-0008D-r3mix-r2continue-r16-a100"; INIT_MODE = "continue"; SEED = 20260825
MAX_SEQ_LENGTH = 2048; MICRO_BATCH = 8; GRAD_ACCUM = 6; EPOCHS = 0.5; LR = 7e-6
LORA_R = 16; LORA_ALPHA = 32; LORA_DROPOUT = 0.0
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
SYSTEM_PROMPT = "You are a helpful assistant that solves math problems step by step."
USER_SUFFIX = "Solve this step by step, then give the final answer as a single integer inside \boxed{}."
HYPOTHESIS = "continued best R2 Pro4-hint r16 trained on R1 + Pro4-hint R2 + R3 Qwen-only strict traces improves fixed-dev SC16 without external inference."

def compact_name(v): return re.sub(r"[\s_-]+", "", unicodedata.normalize("NFC",str(v)).casefold())
def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()
def read_csv(path):
    f=pd.read_csv(path,dtype=str,keep_default_na=False);f.columns=[x.strip() for x in f.columns];return f
def find_one(pattern, must_contain=()):
    found=[]
    for p in RUNS_DIR.glob(pattern):
        if p.is_file() and all(term in str(p) for term in must_contain): found.append(p)
    assert found, (pattern,must_contain)
    return max(found,key=lambda p:p.stat().st_size)

m=Path("/content/drive")
if not (m/"MyDrive").exists(): drive.mount(str(m))
roots=[p for p in (m/"MyDrive").iterdir() if p.is_dir() and compact_name(p.name)==compact_name("2026소중한챌린지")]
assert len(roots)==1,roots
PROJECT_DIR=roots[0]; RUNS_DIR=PROJECT_DIR/"runs"
SPLIT_DIR=RUNS_DIR/"AUDIT-0002-clean-split-passN-20260821-215706"/"splits"
assert (SPLIT_DIR/"split_manifest.json").exists(), SPLIT_DIR
R1_PATH=find_one("RFT-0002-r1-native-k4-full/data/r1_native_verified*.csv")
R2_PATH=find_one("RFT-0003A-pro4-hint-qwen-generation/data/r2_pro4_hint_qwen_65_anchor35_train.csv")
R3_PATH=RUNS_DIR/"RFT-0008B-r3-pro4hint-r2solver-k4"/"data"/"r3_pro4hint_r2solver_verified.csv"
assert R3_PATH.exists(), R3_PATH
PARENT_ADAPTER=(RUNS_DIR/"RFT-0004B-r2-pro4-hint-lowdrift-lora"/"adapter_final") if INIT_MODE=="continue" else None
if PARENT_ADAPTER:
    assert (PARENT_ADAPTER/"adapter_config.json").exists(), PARENT_ADAPTER
    parent_cfg=json.loads((PARENT_ADAPTER/"adapter_config.json").read_text()); parent_base=str(parent_cfg.get("base_model_name_or_path","")).rstrip("/")
    assert parent_base==BASE_MODEL or parent_base.endswith("/Qwen2.5-3B-Instruct"),parent_base
    PARENT_WEIGHT=next((PARENT_ADAPTER/n for n in ["adapter_model.safetensors","adapter_model.bin"] if (PARENT_ADAPTER/n).exists()),None);assert PARENT_WEIGHT
else: PARENT_WEIGHT=None
EXP_DIR=RUNS_DIR/RUN_ID; DATA_DIR=EXP_DIR/"data"; CKPT_DIR=EXP_DIR/"checkpoints"; REPORT_DIR=EXP_DIR/"reports"; TB_DIR=EXP_DIR/"logs"/"tensorboard"; WANDB_DIR=EXP_DIR/"logs"/"wandb"; FINAL_ADAPTER=EXP_DIR/"adapter_final"
for p in [DATA_DIR,CKPT_DIR,REPORT_DIR,TB_DIR,WANDB_DIR]:p.mkdir(parents=True,exist_ok=True)
os.environ["WANDB_DIR"]=str(WANDB_DIR)
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "A100 BF16 runtime required"
print("[GPU]",torch.cuda.get_device_name(0));print("[R1]",R1_PATH);print("[R2]",R2_PATH);print("[R3]",R3_PATH);print("[INIT]",INIT_MODE,PARENT_ADAPTER or BASE_MODEL)

In [ ]:
# Cell 3 — Build the immutable R1 + R2 + R3 corpus. R2 anchors are excluded because R1 supplies them.
BOX_RE=re.compile(r"\\boxed\s*\{\s*(-?\d(?:[\d,]*\d)?)\s*\}")
def norm_int(v):
    v=str(v or "").strip().replace(",","");return str(int(v)) if re.fullmatch(r"-?\d+",v) else None
def terminal_boxed(v):
    text=str(v or "");hits=list(BOX_RE.finditer(text))
    if not hits:return None
    h=hits[-1];tail=re.sub(r"^(?:\\\)|\\\]|\$)+","",text[h.end():].strip());tail=re.sub(r"^[.!]+$","",tail).strip()
    return norm_int(h.group(1)) if not tail else None
def tmpl(q):
    q=re.sub(r"\s+"," ",unicodedata.normalize("NFKC",str(q)).casefold()).strip();return re.sub(r"\d+(?:\.\d+)?","#",q)
def stable_rank(frame, salt):
    frame=frame.copy();frame["_rank"]=frame.apply(lambda x:hashlib.sha256(f"{SEED}|{salt}|{x['id']}|{x['solution']}".encode()).hexdigest(),axis=1);return frame
r1,r2,r3=[read_csv(p) for p in [R1_PATH,R2_PATH,R3_PATH]]
for f,name in [(r1,"r1"),(r2,"r2"),(r3,"r3")]:
    assert {"id","question","answer","solution"}.issubset(f.columns),(name,f.columns)
    f["answer"]=f["answer"].map(norm_int);assert f["answer"].notna().all(),name
    assert all(terminal_boxed(s)==a for s,a in zip(f.solution,f.answer)),name
    f["template_key"]=f.question.map(tmpl)
# one stable native anchor per R1 question; retain up to two independent Qwen rewrite paths for R2/R3.
r1=stable_rank(r1,"r1").sort_values(["id","_rank"]).drop_duplicates("id").drop(columns="_rank");r1["source"]="r1_native_anchor"
if "source" in r2.columns:r2=r2[~r2.source.str.contains("anchor",case=False,na=False)].copy()
r2=stable_rank(r2,"r2").sort_values(["id","_rank"]).groupby("id",group_keys=False).head(2).drop(columns="_rank");r2["source"]="r2_pro4hint_qwen_rewrite"
r3=stable_rank(r3,"r3").sort_values(["id","_rank"]).groupby("id",group_keys=False).head(2).drop(columns="_rank");r3["source"]="r3_pro4hint_qwen_rewrite"
held=[read_csv(SPLIT_DIR/f"{n}_v1.csv") for n in ["tune","dev","test"]];hold_ids=set().union(*(set(x.id) for x in held));hold_tpl=set().union(*(set(x.question.map(tmpl)) for x in held))
mix=pd.concat([r1,r2,r3],ignore_index=True,sort=False)[["id","question","answer","solution","source","template_key"]].drop_duplicates(["id","solution"])
assert not set(mix.id)&hold_ids and not set(mix.template_key)&hold_tpl
assert mix.answer.str.fullmatch(r"-?\d+").all() and all(terminal_boxed(s)==a for s,a in zip(mix.solution,mix.answer))
mix=stable_rank(mix,"mix").sort_values("_rank").drop(columns="_rank").reset_index(drop=True)
TRAIN_CSV=DATA_DIR/"r1_r2_pro4hint_r3_strict_train.csv";mix.to_csv(TRAIN_CSV,index=False,encoding="utf-8")
composition={"r1_rows":len(r1),"r2_rows":len(r2),"r3_rows":len(r3),"final_rows":len(mix),"questions":int(mix.id.nunique()),"source_counts":mix.source.value_counts().to_dict(),"sha256":sha256_file(TRAIN_CSV)}
(REPORT_DIR/"data_manifest.json").write_text(json.dumps(composition,ensure_ascii=False,indent=2),encoding="utf-8")
print("[MIX]",json.dumps(composition,ensure_ascii=False,indent=2));print("[TRAIN CSV]",TRAIN_CSV)

In [ ]:
# Cell 4 — Right-padded assistant-only tokens and a diagnostic loss split.
from datasets import Dataset
from transformers import AutoTokenizer, set_seed

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL, revision=MODEL_REVISION, use_fast=True, token=False
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "right"

def user_text(question):
    return f"{str(question).strip()}\n\n{USER_SUFFIX}"

def prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text(question)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def full(question, solution):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text(question)},
        {"role": "assistant", "content": str(solution).strip()},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

def tokenize(ex):
    prefix = tokenizer(
        prompt(ex["question"]), add_special_tokens=False
    )["input_ids"]
    encoded = tokenizer(
        full(ex["question"], ex["solution"]),
        add_special_tokens=False,
    )
    assert encoded["input_ids"][:len(prefix)] == prefix
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": (
            [-100] * len(prefix) + encoded["input_ids"][len(prefix):]
        ),
        "n_tokens": len(encoded["input_ids"]),
    }

raw = Dataset.from_pandas(
    mix[["id", "question", "answer", "solution", "source"]],
    preserve_index=False,
)
tokenized = raw.map(tokenize, num_proc=2, desc="assistant-only tokenize")

keep = [i for i, n in enumerate(tokenized["n_tokens"]) if n <= MAX_SEQ_LENGTH]
reject = [i for i, n in enumerate(tokenized["n_tokens"]) if n > MAX_SEQ_LENGTH]
assert len(reject) / max(len(tokenized), 1) <= 0.02, (
    len(reject), len(tokenized)
)

mix.iloc[reject].assign(
    total_tokens=[tokenized[i]["n_tokens"] for i in reject]
).to_csv(DATA_DIR / "overlength_reject.csv", index=False)

eligible = tokenized.select(keep)
qids = sorted(
    set(eligible["id"]),
    key=lambda x: hashlib.sha256(
        f"{SEED}|diagnostic|{x}".encode()
    ).hexdigest(),
)
eval_ids = set(qids[:min(256, max(1, len(qids) // 20))])

train_idx = [i for i, qid in enumerate(eligible["id"]) if qid not in eval_ids]
eval_idx = [i for i, qid in enumerate(eligible["id"]) if qid in eval_ids]

remove = ["id", "question", "answer", "solution", "source", "n_tokens"]
train_dataset = eligible.select(train_idx).remove_columns(remove)
eval_dataset = eligible.select(eval_idx).remove_columns(remove)

token_report = {
    "eligible_rows": len(eligible),
    "train_rows": len(train_dataset),
    "diagnostic_eval_rows": len(eval_dataset),
    "overlength_rejected": len(reject),
    "max_tokens": max(eligible["n_tokens"]),
    "train_csv_sha256": sha256_file(TRAIN_CSV),
}
(REPORT_DIR / "token_report.json").write_text(
    json.dumps(token_report, indent=2), encoding="utf-8"
)
print("[TOKENS]", token_report)

In [ ]:
# Cell 5 — Load the selected initial state and create/train the r16 solver adapter.
from transformers import AutoModelForCausalLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments, TrainerCallback
from peft import LoraConfig, PeftModel, get_peft_model

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=MODEL_REVISION, torch_dtype=torch.bfloat16,
    device_map={"": 0}, token=False,
)
if INIT_MODE == "fresh":
    model = get_peft_model(base, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM",
        target_modules=TARGET_MODULES,
    ))
    parent_sha = None
else:
    model = PeftModel.from_pretrained(base, str(PARENT_ADAPTER), is_trainable=True)
    parent_sha = sha256_file(PARENT_WEIGHT)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.train()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
assert 0 < trainable / total < 0.02, (trainable, total)
print(f"[MODEL] init={INIT_MODE} trainable={trainable:,}/{total:,} ({100*trainable/total:.3f}%)")

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8)

class MemoryLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and torch.cuda.is_available():
            free, _ = torch.cuda.mem_get_info()
            logs.update({
                "gpu/allocated_gib": round(torch.cuda.memory_allocated()/1024**3, 3),
                "gpu/reserved_gib": round(torch.cuda.memory_reserved()/1024**3, 3),
                "gpu/max_allocated_gib": round(torch.cuda.max_memory_allocated()/1024**3, 3),
                "gpu/free_gib": round(free/1024**3, 3),
            })
            print("[TRAIN-LOG]", json.dumps(logs), flush=True)
        return control

eval_arg = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
total_updates = max(1, math.ceil(len(train_dataset) / (MICRO_BATCH * GRAD_ACCUM) * EPOCHS))
checkpoint_steps = max(1, round(total_updates / 4))
train_args = {
    "output_dir": str(CKPT_DIR), "logging_dir": str(TB_DIR),
    "num_train_epochs": EPOCHS, "learning_rate": LR,
    "per_device_train_batch_size": MICRO_BATCH, "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "gradient_checkpointing": True, "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "adamw_torch", "lr_scheduler_type": "cosine", "warmup_ratio": 0.03,
    "weight_decay": 0.01, "max_grad_norm": 1.0,
    "bf16": True, "fp16": False, "tf32": True,
    "logging_steps": 10, "logging_first_step": True,
    "save_strategy": "steps", "save_steps": checkpoint_steps, "save_total_limit": 5,
    "load_best_model_at_end": False, "prediction_loss_only": True,
    "report_to": ["tensorboard", "wandb"], "run_name": RUN_ID,
    "remove_unused_columns": False, "group_by_length": True,
    "dataloader_num_workers": 2, "seed": SEED, "data_seed": SEED,
}
train_args[eval_arg] = "steps"; train_args["eval_steps"] = checkpoint_steps
trainer = Trainer(model=model, args=TrainingArguments(**train_args), train_dataset=train_dataset, eval_dataset=eval_dataset, data_collator=collator, callbacks=[MemoryLogger()])
from transformers.trainer_utils import get_last_checkpoint
resume = get_last_checkpoint(str(CKPT_DIR))
print("[TRAIN]", {"init": INIT_MODE, "updates": total_updates, "checkpoint_steps": checkpoint_steps, "resume": resume})
started = time.time(); result = trainer.train(resume_from_checkpoint=resume); seconds = time.time() - started
trainer.save_model(str(FINAL_ADAPTER)); tokenizer.save_pretrained(str(FINAL_ADAPTER))
report = {
    "run_id": RUN_ID, "status": "completed", "hypothesis": HYPOTHESIS,
    "base_model": BASE_MODEL, "model_revision": MODEL_REVISION,
    "init_mode": INIT_MODE, "parent_adapter": str(PARENT_ADAPTER) if PARENT_ADAPTER else None,
    "parent_adapter_sha256": parent_sha, "training_csv": str(TRAIN_CSV), "training_csv_sha256": sha256_file(TRAIN_CSV),
    "rows": {"train": len(train_dataset), "diagnostic_eval": len(eval_dataset)},
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "target_modules": TARGET_MODULES},
    "training": {"epochs": EPOCHS, "lr": LR, "micro_batch": MICRO_BATCH, "gradient_accumulation": GRAD_ACCUM, "effective_batch": MICRO_BATCH*GRAD_ACCUM, "max_seq_length": MAX_SEQ_LENGTH, "assistant_only_loss": True, "bf16": True},
    "runtime_seconds": seconds, "train_metrics": result.metrics,
    "adapter_final": str(FINAL_ADAPTER), "checkpoints": str(CKPT_DIR), "tensorboard": str(TB_DIR), "wandb_offline": str(WANDB_DIR),
    "selection_rule": "Evaluate all four checkpoints on fixed dev SC16; do not select on diagnostic eval loss.",
}
REPORT_PATH = REPORT_DIR / "training_report.json"
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("[TRAIN] adapter:", FINAL_ADAPTER); print("[REPORT]", REPORT_PATH)

In [ ]:
# Cell 6 — Optional GPU runtime release. Set True only after adapter/report artifacts are safely on Drive.
DISCONNECT_GPU_RUNTIME = True
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained. Set DISCONNECT_GPU_RUNTIME=True to release the GPU.")